# Importing necessary Libraries

In [1]:
import pandas as pd

# Step 1: Data Understanding & Preprocessing

## Loading Data

In [2]:
df = pd.read_excel("./Data/REPORT.xlsx")

In [3]:
df

,YEAR,BARCODE,NAME,DIV,DEP,LOC,FE,STATUS,LEAVE DATE,TYPE,NO DAYS
0,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-08-20,AL,1.0
1,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-08-28,AL,1.0
2,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-07-06,AL,0.5
3,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-09-08,AL,1.0
4,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-09-09,AL,1.0
...,...,...,...,...,...,...,...,...,...,...,...
257766,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-24,AL,1.0
257767,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-25,AL,1.0
257768,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-26,AL,1.0
257769,2025.0,5356.0,P.D.C. PREMACHANDRA,DPR,SSO,CAR,2.0,ACTIVE,2025-02-24,AL,0.5


## Handle missing values

In [4]:
df.isnull().sum()

YEAR          14822
BARCODE       14822
NAME          14822
DIV           14822
DEP           14822
LOC           14822
FE            14822
STATUS        14822
LEAVE DATE    14822
TYPE          19311
NO DAYS       14822
dtype: int64

### Handling the missing values of the `LEAVE TYPE` column using the mode of each employee

In [5]:
# Drop rows where 'NAME' is NaN
df = df.dropna(subset=['NAME'])

In [6]:
df.isnull().sum()

YEAR             0
BARCODE          0
NAME             0
DIV              0
DEP              0
LOC              0
FE               0
STATUS           0
LEAVE DATE       0
TYPE          4489
NO DAYS          0
dtype: int64

In [7]:
# Compute the mode of leave_type for each employee
mode_per_employee = df.groupby('NAME')['TYPE'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

# Fill null values in leave_type using the computed mode
df['TYPE'] = df.apply(lambda row: mode_per_employee[row['NAME']] if pd.isna(row['TYPE']) else row['TYPE'], axis=1)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13904\893224361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TYPE'] = df.apply(lambda row: mode_per_employee[row['NAME']] if pd.isna(row['TYPE']) else row['TYPE'], axis=1)


In [8]:
df.isnull().sum()

YEAR          0
BARCODE       0
NAME          0
DIV           0
DEP           0
LOC           0
FE            0
STATUS        0
LEAVE DATE    0
TYPE          0
NO DAYS       0
dtype: int64

In [9]:
df.nunique()

YEAR             6
BARCODE       1563
NAME          1562
DIV             13
DEP             41
LOC             93
FE               4
STATUS           1
LEAVE DATE    1832
TYPE            17
NO DAYS          2
dtype: int64

In [10]:
df = df.drop(columns=['YEAR', 'FE', 'STATUS'])

## Feature Engineering

### Converting "LEAVE DATE" to datatime format

In [11]:
df['LEAVE DATE'] = pd.to_datetime(df['LEAVE DATE'], errors='coerce')

In [12]:
df.describe()

,BARCODE,LEAVE DATE,NO DAYS
count,242949.000000,242949,242949.000000
mean,3406.992682,2022-10-27 03:03:22.486941696,0.845636
min,26.000000,2019-01-30 00:00:00,0.500000
25%,2799.000000,2021-08-19 00:00:00,0.500000
50%,3363.000000,2022-11-05 00:00:00,1.000000
75%,4099.000000,2024-01-27 00:00:00,1.000000
max,5357.000000,2025-06-04 00:00:00,1.000000
std,962.564840,NaN,0.230985


In [13]:
# Filter out rows with LEAVE YEAR as 2025
#df = df[df['LEAVE DATE'].dt.year != 2025]

In [14]:
# Check unique years remaining in the dataframe
print(df['LEAVE DATE'].dt.year.unique())

[2020 2021 2022 2019 2023 2024 2025]


### Extract Month, Day of the Week, and Year

In [15]:
df['LEAVE MONTH'] = df['LEAVE DATE'].dt.month
df['LEAVE YEAR'] = df['LEAVE DATE'].dt.year
df['LEAVE DAY OF WEEK'] = df['LEAVE DATE'].dt.dayofweek

```python
df['LEAVE DATE'].dt.month
```
- Extracts the month from the LEAVE DATE column.
- Returns the month as a number (e.g., 1 for January, 2 for February, etc.). 

```python
df['LEAVE DATE'].dt.dayofweek
```
- Extracts the day of the week from the LEAVE DATE column.
- Returns the day of the week as a number, where:
    - 0 = Monday
    - 1 = Tuesday
    - 2 = Wednesday
    - 3 = Thursday
    - 4 = Friday
    - 5 = Saturday
    - 6 = Sunday

In [16]:
df

,BARCODE,NAME,DIV,DEP,LOC,LEAVE DATE,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK
0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,2020-08-20,AL,1.0,8,2020,3
1,26.0,M.G.R.LIVERA,DYA,SMT,PP4,2020-08-28,AL,1.0,8,2020,4
2,26.0,M.G.R.LIVERA,DYA,SMT,PP4,2020-07-06,AL,0.5,7,2020,0
3,26.0,M.G.R.LIVERA,DYA,SMT,PP4,2020-09-08,AL,1.0,9,2020,1
4,26.0,M.G.R.LIVERA,DYA,SMT,PP4,2020-09-09,AL,1.0,9,2020,2
...,...,...,...,...,...,...,...,...,...,...,...
257766,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2025-02-24,AL,1.0,2,2025,0
257767,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2025-02-25,AL,1.0,2,2025,1
257768,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2025-02-26,AL,1.0,2,2025,2
257769,5356.0,P.D.C. PREMACHANDRA,DPR,SSO,CAR,2025-02-24,AL,0.5,2,2025,0


### Calculate Time Gap Between Leaves

In [17]:
# Sort data by employee and leave date
df = df.sort_values(by=['NAME', 'LEAVE DATE'])

#### Calculating days since last leave

In [18]:
# Compute difference between consecutive leaves per employee
df['PREV LEAVE DATE'] = df.groupby('NAME')['LEAVE DATE'].shift(1)
df['DAYS SINCE LAST LEAVE'] = (df['LEAVE DATE'] - df['PREV LEAVE DATE']).dt.days

#### Calculating days until next leave

In [19]:
df['NEXT LEAVE DATE'] = df.groupby('NAME')['LEAVE DATE'].shift(-1)
df['DAYS UNTIL NEXT LEAVE'] = (df['NEXT LEAVE DATE'] - df['LEAVE DATE']).dt.days

In [20]:
df

,BARCODE,NAME,DIV,DEP,LOC,LEAVE DATE,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE
6202,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-11,AL,0.5,1,2020,5,NaT,NaN,2020-01-31,20.0
6213,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-31,CS,1.0,1,2020,4,2020-01-11,20.0,2020-02-01,1.0
6212,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-01,CS,0.5,2,2020,5,2020-01-31,1.0,2020-02-15,14.0
6204,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-15,AL,0.5,2,2020,5,2020-02-01,14.0,2020-02-18,3.0
6215,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-18,CS,1.0,2,2020,1,2020-02-15,3.0,2020-03-10,21.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248233,5216.0,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2024-11-14,DL,1.0,11,2024,3,2024-10-08,37.0,2025-01-16,63.0
257338,5216.0,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2025-01-16,SK,1.0,1,2025,3,2024-11-14,63.0,2025-01-20,4.0
257336,5216.0,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2025-01-20,SK,1.0,1,2025,0,2025-01-16,4.0,2025-01-30,10.0
257334,5216.0,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2025-01-30,CS,1.0,1,2025,3,2025-01-20,10.0,2025-02-05,6.0


#### Calculating the cumulative leave count for each year

In [21]:
# Sort by employee and leave_date
df = df.sort_values(by=['NAME', 'LEAVE YEAR', 'LEAVE DATE'])

# Compute cumulative leave count per year for each employee
df['CUMULATIVE LEAVE COUNT'] = df.groupby(['NAME', 'LEAVE YEAR']).cumcount() + 1


- `CUMULATIVE LEAVE COUNT` resets every new year for each employee.

In [22]:
# Print an example for an employee to verify the answer
df[df['NAME'] == "A.A.A.R.DE SILVA"]

,BARCODE,NAME,DIV,DEP,LOC,LEAVE DATE,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
6202,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-11,AL,0.5,1,2020,5,NaT,NaN,2020-01-31,20.0,1
6213,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-31,CS,1.0,1,2020,4,2020-01-11,20.0,2020-02-01,1.0,2
6212,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-01,CS,0.5,2,2020,5,2020-01-31,1.0,2020-02-15,14.0,3
6204,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-15,AL,0.5,2,2020,5,2020-02-01,14.0,2020-02-18,3.0,4
6215,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-18,CS,1.0,2,2020,1,2020-02-15,3.0,2020-03-10,21.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193746,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-12-13,AL,1.0,12,2024,4,2024-11-25,18.0,2024-12-23,10.0,23
193740,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-12-23,AL,1.0,12,2024,0,2024-12-13,10.0,2024-12-24,1.0,24
193741,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-12-24,AL,1.0,12,2024,1,2024-12-23,1.0,2024-12-31,7.0,25
193738,2673.0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-12-31,AL,1.0,12,2024,1,2024-12-24,7.0,2025-01-06,6.0,26


Getting unique values of the `LEAVE TYPE` column

In [23]:
unique_values = df['TYPE'].unique()
unique_values

array(['AL', 'CS', 'SK', 'SP', 'QU', 'CP', 'DL', 'SD', 'MA', 'TD', 'LD',
       'AC', 'SU', 'SS', 'ML', 'IN', 'MT'], dtype=object)

### Calculates the leave type distribution per employee

In [24]:
# # Get total leaves per employee
# total_leaves_per_employee = df.groupby('SERVICE NO')['LEAVE TYPE'].count()
# 
# # Get leave type counts per employee
# leave_type_distribution = df.groupby(['SERVICE NO', 'LEAVE TYPE']).size().unstack(fill_value=0)
# 
# # Convert counts to proportions
# leave_type_distribution = leave_type_distribution.div(total_leaves_per_employee, axis=0)
# 
# # Merge with main dataframe
# df = df.merge(leave_type_distribution, on='SERVICE NO', how='left')
# df

In [25]:
df = df.drop(columns=['BARCODE'])

In [26]:
df_with_next_leave = df[df['NEXT LEAVE DATE'].notnull()]
df_with_no_next_leave = df[df['NEXT LEAVE DATE'].isnull()]

In [27]:
df_with_next_leave.isnull().sum()

NAME                         0
DIV                          0
DEP                          0
LOC                          0
LEAVE DATE                   0
TYPE                         0
NO DAYS                      0
LEAVE MONTH                  0
LEAVE YEAR                   0
LEAVE DAY OF WEEK            0
PREV LEAVE DATE           1548
DAYS SINCE LAST LEAVE     1548
NEXT LEAVE DATE              0
DAYS UNTIL NEXT LEAVE        0
CUMULATIVE LEAVE COUNT       0
dtype: int64

In [28]:
df_with_next_leave = df_with_next_leave.dropna()

In [29]:
df_with_next_leave.isnull().sum()

NAME                      0
DIV                       0
DEP                       0
LOC                       0
LEAVE DATE                0
TYPE                      0
NO DAYS                   0
LEAVE MONTH               0
LEAVE YEAR                0
LEAVE DAY OF WEEK         0
PREV LEAVE DATE           0
DAYS SINCE LAST LEAVE     0
NEXT LEAVE DATE           0
DAYS UNTIL NEXT LEAVE     0
CUMULATIVE LEAVE COUNT    0
dtype: int64

In [30]:
df_with_next_leave.to_csv('./Data/after_preprocess.csv', index=False)

In [31]:
df_with_no_next_leave.isnull().sum()

NAME                         0
DIV                          0
DEP                          0
LOC                          0
LEAVE DATE                   0
TYPE                         0
NO DAYS                      0
LEAVE MONTH                  0
LEAVE YEAR                   0
LEAVE DAY OF WEEK            0
PREV LEAVE DATE             14
DAYS SINCE LAST LEAVE       14
NEXT LEAVE DATE           1562
DAYS UNTIL NEXT LEAVE     1562
CUMULATIVE LEAVE COUNT       0
dtype: int64

In [32]:
df_with_no_next_leave = df_with_no_next_leave.dropna(subset=['PREV LEAVE DATE', 'DAYS SINCE LAST LEAVE'])

In [33]:
df_with_no_next_leave.isnull().sum()

NAME                         0
DIV                          0
DEP                          0
LOC                          0
LEAVE DATE                   0
TYPE                         0
NO DAYS                      0
LEAVE MONTH                  0
LEAVE YEAR                   0
LEAVE DAY OF WEEK            0
PREV LEAVE DATE              0
DAYS SINCE LAST LEAVE        0
NEXT LEAVE DATE           1548
DAYS UNTIL NEXT LEAVE     1548
CUMULATIVE LEAVE COUNT       0
dtype: int64

In [34]:
df_with_no_next_leave.to_csv('./Data/data_without_next_leave.csv', index=False)